# Text Moderation — TF toxicity classifier (Reddit IRL + lexicons)

Labels a comment toxic/clean for `app/moderation_engine.py`
`/api/v1/moderation/text`. Ground truth is bootstrapped from three local corpora:
the Reddit r/IRL comment stream (`the-reddit-irl-dataset-comments.csv`), the
profanity lexicon (`profanity_en.csv`), and Gen-Z slang intensity
(`genz_slang_usage_2020_2025.csv`). Jigsaw (HF `oxford-ds/toxicity`) is the gold
external source to swap in when an internet connection is available. The trained
model is exported as an ONNX artifact (token-ID input) plus the persisted
text-vectorizer vocabulary, so the serving layer can reproduce tokenisation.

In [1]:
import importlib.util
import os
import pathlib
import sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'tensorflow_text'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


2026-08-05 10:07:39.860652: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-05 10:07:40.111222: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-05 10:07:44.469102: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF 2.20.0 | GPU: False | scale: demo


2026-08-05 10:07:48.239465: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Data + bootstrapped labels (lexicon/harsh-slang prior)
import os
import numpy as np
import pandas as pd
from buddy_data import profanity, genz_slang

N = {'smoke': 4_000, 'demo': 80_000, 'full': 500_000}[SCALE]
USE_SLANG = os.environ.get('BUDDY_SLANG', '1') == '1'

prof = profanity()
slang = genz_slang()

# 2.3 GB CSV — stream only the body column in chunks (bounded memory)
from pathlib import Path
rng = np.random.default_rng(42)
chunks = pd.read_csv(Path('../data/the-reddit-irl-dataset-comments.csv'),
                     usecols=['body'], chunksize=250_000, low_memory=False)
texts, want = [], N
for ci, chunk in enumerate(chunks):
    col = chunk['body'].dropna().astype(str)
    texts.append(col.sample(n=min(want, len(col)), random_state=42 + ci))
    want -= len(texts[-1])
    if want <= 0:
        break
texts = pd.concat(texts).tolist()[:N]
print('sampled IRL bodies:', len(texts))

bad = set()
for col in ('canonical_form_1', 'canonical_form_2', 'canonical_form_3'):
    bad |= {w.lower() for w in prof[col].dropna().astype(str)}
hard_slang = set(
    slang.loc[(slang['sentiment_score'] <= -0.2) | (slang['intensity_score'] >= 0.8),
              'slang_term'].dropna().str.lower()
)

def to_label(s: str) -> float:
    s = s.lower()
    if any(w in s for w in bad):
        return 1.0
    if USE_SLANG and any(w in s for w in hard_slang):
        return 0.8
    return 0.0

df = pd.DataFrame({'text': texts})
df['label'] = df['text'].map(to_label)
print('n =', len(df), '| label dist:', df['label'].value_counts().round(3).to_dict())
print('| toxic rate:', round((df['label'] > 0).mean(), 4))

sampled IRL bodies: 80000
n = 80000 | label dist: {0.0: 54401, 1.0: 18126, 0.8: 7473}
| toxic rate: 0.32


In [3]:
# Hold-out split (natural skew for honest AUC) + balanced train set
from sklearn.model_selection import train_test_split
train, val = train_test_split(df, test_size=0.2, random_state=42, stratify=(df['label'] > 0))

pos = train[train['label'] > 0]
neg_pool = train[train['label'] == 0]
neg = neg_pool.sample(n=min(len(pos) * 4, len(neg_pool)), random_state=42)
train = pd.concat([pos, neg]).sample(frac=1, random_state=42)
print('train:', train.shape, 'toxic frac:', round((train['label'] > 0).mean(), 3),
      '| val:', val.shape, 'toxic frac:', round((val['label'] > 0).mean(), 4))

train: (64000, 2) toxic frac: 0.32 | val: (16000, 2) toxic frac: 0.32


In [4]:
# Vectorizer + BiLSTM classifier (TF/Keras)
import tensorflow as tf

MAX_TOKENS, SEQ, EMB, HID = 40_000, 128, 96, 48
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_sequence_length=SEQ, standardize='lower_and_strip_punctuation')
vectorizer.adapt(np.array(train['text']))
print('vocab size:', vectorizer.vocabulary_size())

inp = tf.keras.Input(shape=(SEQ,), dtype='int64')
x = tf.keras.layers.Embedding(MAX_TOKENS + 2, EMB)(inp)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(HID, return_sequences=True))(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
m.summary()

vocab size: 40000


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 128, 96)        │     3,840,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128, 96)        │        55,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 96)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,895,969 (14.86 MB)

 Trainable params: 3,895,969 (14.86 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train (tokenize once, batch on CPU)
import time
Xtr = vectorizer(np.array(train['text'])).numpy()
Ytr = train['label'].to_numpy()
Xv = vectorizer(np.array(val['text'])).numpy()
Yv = val['label'].to_numpy()

EPOCHS = {'smoke': 1, 'demo': 3, 'full': 6}[SCALE]
t0 = time.time()
hist = m.fit(Xtr, Ytr, epochs=EPOCHS, batch_size=256,
             validation_data=(Xv, Yv), verbose=1)
print(f'train {time.time()-t0:.0f}s')

Epoch 1/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 186s 718ms/step - accuracy: 0.7323 - loss: 0.4923 - val_accuracy: 0.8236 - val_loss: 0.3405
Epoch 2/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 233s 933ms/step - accuracy: 0.8703 - loss: 0.2054 - val_accuracy: 0.8822 - val_loss: 0.1613
Epoch 3/3
250/250 ━━━━━━━━━━━━━━━━━━━━ 232s 926ms/step - accuracy: 0.8943 - loss: 0.1090 - val_accuracy: 0.8731 - val_loss: 0.1560
train 652s


In [6]:
# Evaluate: AUC / average precision + threshold calibration (F1)
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

Yb = (Yv > 0).astype(int)                  # metrics need binary labels (0.8 = soft)
p = m.predict(Xv, batch_size=512)[:, 0]
if Yb.sum() == 0 or len(np.unique(Yb)) < 2:
    print('WARNING: val split has no positives (natural skew). Metrics undefined — '
          'raise N / BUDDY_SCALE for real signal.')
    auc = ap = float('nan'); best, f1_50 = 0.5, float('nan')
else:
    auc = roc_auc_score(Yb, p)
    ap = average_precision_score(Yb, p)
    prec, rec, thr = precision_recall_curve(Yb, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    best = float(thr[np.argmax(f1[:-1])])
    f1_50 = f1_score(Yb, (p > 0.5).astype(int))
    print(f'val AUC={auc:.3f} AP={ap:.3f}')
    print(f'@0.5      P={precision_score(Yb,(p>0.5).astype(int)):.3f} '
          f'R={recall_score(Yb,(p>0.5).astype(int)):.3f} F1={f1_50:.3f}')
    print(f'@F1-best  threshold={best:.3f} '
          f'F1={f1_score(Yb,(p>best).astype(int)):.3f}')

32/32 ━━━━━━━━━━━━━━━━━━━━ 19s 548ms/step
val AUC=0.987 AP=0.982
@0.5      P=0.921 R=0.943 F1=0.932
@F1-best  threshold=0.736 F1=0.948


In [7]:
# Check agreement against a held-out lexicon test (sanity)
import json
probe = pd.DataFrame({
    'text': ['i love this recipe, thanks!', 'shut up you stupid idiot', 'great post!',
             'this is the worst thing ive ever seen', 'hahaha lol'],
})
Xp = vectorizer(np.array(probe['text'])).numpy()
probe['score'] = m.predict(Xp)[:, 0].round(3)
print(probe.to_string(index=False))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
                                 text  score
          i love this recipe, thanks!  0.003
             shut up you stupid idiot  0.003
                          great post!  0.003
this is the worst thing ive ever seen  0.003
                           hahaha lol  0.003


### Export contract (consumed by the AI service)

The cells below write `../models/toxicity_classifier.onnx` and its dynamic-INT8 quantized copy
`toxicity_classifier_int8.onnx`. `app/ml/serving.py::load_preferred('toxicity_classifier')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [8]:
# Export ONNX (+ INT8) + persist the vectorizer vocab for serving
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(
    m, Path('../models'), 'toxicity_classifier', '1.0.0',
    input_signature=[tf.TensorSpec((None, SEQ), tf.int64, name='input_ids')])
q = quantize_dynamic_onnx(onnx)

vocab = {'max_tokens': MAX_TOKENS, 'sequence_length': SEQ,
         'vocabulary': vectorizer.get_vocabulary(), 'threshold': best}
(Path('../models') / 'toxicity_vectorizer.json').write_text(json.dumps(vocab))

mlflow_log({'name': 'toxicity_classifier', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'val_auc': float(auc), 'val_ap': float(ap), 'threshold': float(best)}})
print('exported', q)

Type is unsupported, or the types of the items don't match field type in CollectionDef. Note this is a warning and probably safe to ignore.
'NoneType' object has no attribute 'name'


I0000 00:00:1785914357.038460 1062504 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785914357.038727 1062504 single_machine.cc:376] Starting new session
I0000 00:00:1785914358.693145 1062504 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785914358.693448 1062504 single_machine.cc:376] Starting new session
I0000 00:00:1785914359.698212 1062504 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


{
  "name": "toxicity_classifier",
  "version": "1.0.0",
  "artifact_path": "../models/toxicity_classifier-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "metrics": {
    "val_auc": 0.9869737602682674,
    "val_ap": 0.9815351047847904,
    "threshold": 0.7360104322433472
  }
}
exported ../models/toxicity_classifier-1.0.0_int8.onnx
